In [ ]:
# Libraries
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt

# ASE imports
from ase import units
from ase.units import fs, kB
from ase.io import read, iread, write
from ase.geometry.rdf import get_rdf
from ase.neighborlist import neighbor_list
from ase.geometry import find_mic

# Advanced analysis & Colab
from pymbar import timeseries
from google.colab import drive

#### Useful functions
# reads local OUTCAR file for analysis (see "general analysis" notebook) from a certain index on
def clean_dataset(outcar_file, destination_file, start_idx):
    try:
        atoms_list = read(outcar_file, index=":")
        atoms_list = atoms_list[start_idx:]
        print(f"Retrieved {len(atoms_list)} structures from OUTCAR.")

        # usable file
        write(destination_file, atoms_list, format="extxyz")
        print(f"File saved as {destination_file}")

    except Exception as e:
        print(f"Error cleaning OUTCAR: {e}")

    return 0


# splits data in train and validation for MACE
def split_train_valid(
    full_cleaned_extxyz_file,
    destination_train_file,
    destination_valid_file,
    train_frac,
    seed=1,
):
    traj = ase.io.read(full_cleaned_extxyz_file, index=":")
    print(f"Total frames uploaded: {len(traj)}")

    rng = np.random.RandomState(seed)
    indices = np.arange(len(traj))
    rng.shuffle(indices)
    traj = [traj[i] for i in indices]

    split = int(train_frac * len(traj))
    train_frames = traj[:split]
    valid_frames = traj[split:]

    ase.io.write(destination_train_file, train_frames)
    ase.io.write(destination_valid_file, valid_frames)

    print(
        f"Saved {len(train_frames)} frames in {destination_train_file} and {len(valid_frames)} in {destination_valid_file}"
    )

def check_equilibration(observable, name, dt=1):
    """
    observable: 1D array, one scalar value per frame
    Returns t0 (equilibration frame), g (statistical inefficiency),
    Neff (effective uncorrelated sample count)
    """
    t0, g, Neff = timeseries.detect_equilibration(observable)
    print(f"[{name}] t0 = {t0} (frame {t0*dt}), "
          f"g = {g:.2f}, N_eff = {Neff:.1f} / {len(observable)}")
    return t0, g, Neff



def full_analysis(input_traj_dir, output_dir,rmax=3.9,nbins=1000, timestep_in_fs = 0.5,window_size = 20):
  outcar_file= input_traj_dir #just to not change the name in all the scripts
  images = read(outcar_file, index=':')

  atoms= read(outcar_file)
  cell = images[0].get_cell()


  # analysis selection
  VisCel= True                                      # Visulization of the cell
  RDF_whole= True                                    # RDF analysis
  RDF_firstvslast = True # RDF COMAPARISON FIRST AND LAST PART TRAJECTORY
  TempConv= True         # TEMPERATURE CONVERGENCE
  RDF_peak_fwhm = True # RDF FIRST PEAK AND FWHM VS SIM TIME
  MSD = True     ## MSD relative to initial positions
  EnergyConv= True       #  ENERGY CONVERGENCE
  CompleteAnalysis= True  # complete analysis of rdf coordination number potential etc
  Individual_pair=  True # INDIVIDUAL PAIR ANALYSIS
  ADF = True
  OACF = True
  VDOS = True
  CompleteMSD = True
  TemperatureCrossCheck = True  # TEMPERATURE CROSS CHECK


  ###### Visulization of the cell ###########
  if(VisCel==True):
    a, b, c = cell
    origin = np.zeros(3)

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    # Draw vectors
    for vec, color, label in zip([a, b, c], ['r', 'g', 'b'], ['a', 'b', 'c']):
        ax.quiver(*origin, *vec, color=color, arrow_length_ratio=0.1, linewidth=2, label=label)

    # Draw cell edges
    corners = np.array([
        origin,
        a, b, c,
        a+b, a+c, b+c,
        a+b+c
    ])

    edges = [
        (0,1), (0,2), (0,3),  # from origin
        (1,4), (1,5),  # from a
        (2,4), (2,6),  # from b
        (3,5), (3,6),  # from c
        (4,7), (5,7), (6,7)   # to opposite corner
    ]

    for i, j in edges:
        ax.plot(*zip(corners[i], corners[j]), 'k--', alpha=0.3)

    ax.set_xlabel('X (Å)')
    ax.set_ylabel('Y (Å)')
    ax.set_zlabel('Z (Å)')
    ax.legend()
    ax.set_title('Simulation Cell')
    plt.tight_layout()

    # Save the figure instead of plotting it
    save_path = os.path.join(output_dir, "simulation_cell.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig) # Close the figure to free up memory

    my_full_file=input_traj_dir
    atoms_list = read(my_full_file, index=":")
    os.makedirs(output_dir, exist_ok=True)



  ############ RDF analysis whole traj##############
  if (RDF_whole==True):
    all_rdfs = []
    distances = None


    for atoms in images:
        rdf_values, distances = get_rdf(atoms, rmax=rmax, nbins=nbins) # RDF for EACH individual atom, averaged over all atoms...
        all_rdfs.append(rdf_values)

    avg_rdf = np.mean(all_rdfs, axis=0) # ...averaged over time steps
    fig = plt.figure(figsize=(14,6))
    plt.plot(distances, avg_rdf, color='darkgreen', lw=2, label='Average RDF')
    plt.xlabel('Distance r (Å)')
    plt.ylabel('g(r)')
    plt.title('Radial Distribution Function (all atoms, time-averaged)')
    plt.grid(True)
    plt.legend()

    # Save the figure instead of plotting it
    save_path = os.path.join(output_dir, "average_rdf.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig) # Close the figure to free up memory

  ########## RDF COMAPARISON FIRST AND LAST PART TRAJECTORY #########
  if(RDF_firstvslast==True):
    total_frames = len(images)
    chunk_size = total_frames // 4

    start_rdfs = all_rdfs[:chunk_size]
    end_rdfs = all_rdfs[-chunk_size:]

    avg_rdf_start = np.mean(start_rdfs, axis=0)
    avg_rdf_end = np.mean(end_rdfs, axis=0)
    fig = plt.figure(figsize=(14, 6))
    plt.xlim([0.9,3.9])
    plt.plot(distances, avg_rdf_start, color='blue', lw=2, label='First 25% of Trajectory')
    plt.plot(distances, avg_rdf_end, color='orange', lw=2, label='Last 25% of Trajectory')
    plt.xlabel('Distance r (Å)')
    plt.ylabel('g(r)')
    plt.title('RDF Structural Evolution Comparison')
    plt.grid(True)
    plt.legend()

    # Save the figure instead of plotting it
    save_path = os.path.join(output_dir, "rdf_evolution_comparison.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig) # Close the figure to free up memory


  ######### TEMPERATURE CONVERGENCE ###############

  if(TempConv==True):
    all_velocities = []
    for i in range(len(images) - 1):
        delta_p = images[i+1].get_positions() - images[i].get_positions()
        delta_p, _ = find_mic(delta_p, images[i].get_cell(), pbc=images[i].get_pbc())
        all_velocities.append(delta_p / timestep_in_fs)
    all_velocities = np.array(all_velocities)

    N_atoms = len(images[0])
    dof = 3 * N_atoms - 3   # only remove 3 for center-of-mass translation (see "Temperature" section for more details)

    temperatures = []
    for i, atoms in enumerate(images[:-1]):
        m = atoms.get_masses()
        v_ase = all_velocities[i] / units.fs
        ek = 0.5 * np.sum(m[:, np.newaxis] * v_ase**2)
        temperatures.append((2 * ek) / (dof * units.kB))
    temperatures = np.array(temperatures)
    time_steps=np.arange(len(images))[:-1]
    mean_temperature=np.mean(temperatures)
    fig = plt.figure(figsize=(10, 5))
    plt.plot(time_steps, temperatures, color="crimson", linewidth=1, label="Instantaneous T")
    plt.axhline(y=mean_temperature, color="black", linestyle="--", linewidth=1.5, label=f"Mean T ({mean_temperature:.2f} K)")

    plt.xlabel("Frame Index / Time step")
    plt.ylabel("Temperature (K)")
    plt.title("Temperature Profile (all atoms) Along the MD Trajectory")
    plt.legend()
    plt.grid(True, linestyle=":", alpha=0.6)

    plt.tight_layout()

    # Save the figure instead of plotting it
    save_path = os.path.join(output_dir, "temperature_profile.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig) # Close the figure to free up memory


  ########## RDF FIRST PEAK AND FWHM VS SIM TIME #########
  if(RDF_peak_fwhm== True):
    g_max_history = []
    fwhm_history = []
    time_steps = []

    for start_idx in range(0, total_frames - window_size + 1, window_size):
        window_rdfs = []
        for atoms in images[start_idx : start_idx + window_size]:
            rdf_vals, distances = get_rdf(atoms, rmax=rmax, nbins=nbins)
            window_rdfs.append(rdf_vals)
        window_avg_rdf = np.mean(window_rdfs, axis=0)

        max_idx = np.argmax(window_avg_rdf)
        g_max = window_avg_rdf[max_idx]
        half_max = g_max / 2.0

        left_indices = np.where(window_avg_rdf[:max_idx] < half_max)[0]
        right_indices = np.where(window_avg_rdf[max_idx:] < half_max)[0]
        if len(left_indices) > 0 and len(right_indices) > 0:
            fwhm = distances[max_idx + right_indices[0]] - distances[left_indices[-1]]
        else:
            fwhm = np.nan

        g_max_history.append(g_max)
        fwhm_history.append(fwhm)
        time_steps.append(start_idx + window_size // 2)

    g_max_history = np.array(g_max_history)
    fwhm_history = np.array(fwhm_history)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 8), sharex=True)

    # --- Peak height (g_max) ---
    ax1.plot(time_steps, g_max_history, 'o-', color='blue', lw=1.5, markersize=4)
    ax1.axhline(np.mean(g_max_history[-5:]), color='red', linestyle='--',
                label=f"Mean last 5 windows = {np.mean(g_max_history[-5:]):.3f}") # last 5 windows; window covers 10_000/20= 500 frames; so 500*5 = 2500 frames shown
    ax1.set_ylabel('Peak height g(r)')
    ax1.grid(True)
    ax1.legend()
    ax1.set_title('RDF first‐peak height vs. simulation time')

    # --- FWHM ---
    ax2.plot(time_steps, fwhm_history, 's-', color='orange', lw=1.5, markersize=4)
    ax2.axhline(np.mean(fwhm_history[-5:]), color='red', linestyle='--',
                label=f"Mean last 5 windows = {np.mean(fwhm_history[-5:]):.3f} Å")
    ax2.set_xlabel('MD step (window centre)')
    ax2.set_ylabel('FWHM (Å)')
    ax2.grid(True)
    ax2.legend()
    ax2.set_title('RDF first‐peak FWHM vs. simulation time')

    plt.tight_layout()

    # Save the figure instead of plotting it
    save_path = os.path.join(output_dir, "rdf_peak_evolution.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig) # Close the figure to free up memory


  ########## MSD relative to initial positions #########

  if(MSD==True):
    pbc = images[0].get_pbc()

    # accumulate unwrapped displacement frame-to-frame
    unwrapped_disp = np.zeros_like(images[0].get_positions())
    msd_full_mic = [0.0]

    prev_pos = images[0].get_positions()
    for atoms in images[1:]:
        pos = atoms.get_positions()
        step_disp = pos - prev_pos
        step_disp, _ = find_mic(step_disp, cell, pbc=pbc)
        unwrapped_disp += step_disp
        msd_full_mic.append(np.mean(np.sum(unwrapped_disp**2, axis=1)))
        prev_pos = pos

    msd_full_mic = np.array(msd_full_mic)
    steps = np.arange(len(msd_full_mic))

    fig = plt.figure(figsize=(10, 6))
    plt.plot(steps, msd_full_mic, lw=0.8, alpha=0.6, color='blue', label='Raw MSD')

    plt.xlabel('MD Step')
    plt.ylabel('Mean square displacement (Å²)')
    plt.title('MSD relative to initial positions')
    plt.grid(True, alpha=0.3)
    #plt.legend()
    plt.tight_layout()

    # Save the figure instead of plotting it
    save_path = os.path.join(output_dir, "raw_msd.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig) # Close the figure to free up memory


  ##############  ENERGY CONVERGENCE ################
  if(EnergyConv==True):
    outcar_images=read(outcar_file, index=':')

    energies = np.array([atoms.get_potential_energy() for atoms in outcar_images])
    t0_E, g_E, Neff_E = check_equilibration(energies, "Total Energy", dt=1)
    # Plot energy alongside for visual check
    fig = plt.figure(figsize=(10, 4))
    plt.plot(energies, lw=1, color='darkgreen')
    plt.axvline(t0_E, color='red', linestyle='--', label=f't0 = {t0_E}')
    plt.xlabel('MD step')
    plt.ylabel('Total energy (eV)')
    plt.title('Energy convergence')
    plt.grid(True)
    plt.legend()
    plt.tight_layout()

    # Save the figure instead of plotting it
    save_path = os.path.join(output_dir, "energy_convergence.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig) # Close the figure to free up memory






  ########### PREVIOUS EQUILIBRIUM ANALYSIS -> just used the whole trajectory #########

  ############ complete analysis of rdf coordination number potential etc #########


  images_equilibrium=read(outcar_file, index=':')
  start_eq = 0                                                                    # start equilibrium se to 0 to have on the whole traj
  equilibrium_rdfs = all_rdfs[start_eq:]   # slice the original RDF list

  if(CompleteAnalysis==True):
    # Compute average RDF over equilibrium frames
    avg_rdf_eq = np.mean(equilibrium_rdfs, axis=0)

    # Number density and coordination number (same as before, using equilibrium avg RDF)
    reference_atoms = images_equilibrium[0]
    volume = reference_atoms.get_volume()
    num_target_atoms = len(reference_atoms)
    print("N atoms (equilibrium): ", num_target_atoms, "in volume:", volume)
    number_density = num_target_atoms / volume

    dr = distances[1] - distances[0]
    coordination_integrand = 4 * np.pi * number_density * (distances**2) * avg_rdf_eq
    coordination_number = np.cumsum(coordination_integrand) * dr

    # Potential of Mean Force (PMF)
    with np.errstate(divide='ignore'):
        pmf_eq = -np.log(avg_rdf_eq)
    pmf_eq[np.isinf(pmf_eq)] = np.nan

    # --- Rolling window peak tracking (g_max and FWHM) for equilibrium part ---
    # Use the same window_size, but now on the equilibrium RDFs
    window_size = 20
    total_eq_frames = len(equilibrium_rdfs)
    g_max_eq = []
    fwhm_eq = []
    time_steps_eq = []

    for start_idx in range(0, total_eq_frames - window_size + 1, window_size):
        window_rdfs = equilibrium_rdfs[start_idx : start_idx + window_size]
        window_avg_rdf = np.mean(window_rdfs, axis=0)

        max_idx = np.argmax(window_avg_rdf)
        g_max = window_avg_rdf[max_idx]
        half_max = g_max / 2.0

        left_indices = np.where(window_avg_rdf[:max_idx] < half_max)[0]
        right_indices = np.where(window_avg_rdf[max_idx:] < half_max)[0]
        if len(left_indices) > 0 and len(right_indices) > 0:
            fwhm = distances[max_idx + right_indices[0]] - distances[left_indices[-1]]
        else:
            fwhm = np.nan

        g_max_eq.append(g_max)
        fwhm_eq.append(fwhm)
        time_steps_eq.append(start_idx + window_size // 2)   # relative frame index within equilibrium

    # Convert to numpy arrays
    g_max_eq = np.array(g_max_eq)
    fwhm_eq = np.array(fwhm_eq)
    time_steps_eq = np.array(time_steps_eq)
    fig, axs = plt.subplots(2, 2, figsize=(14, 10))

    axs[0, 0].plot(distances, avg_rdf_eq, color='darkgreen', lw=2)
    axs[1,0].set_xlim([0., 4.])
    axs[0, 0].set_xlabel('Distance r (Å)')
    axs[0, 0].set_ylabel('g(r)')
    axs[0, 0].set_title('Standard Total RDF (All Atoms, time-averaged AT EQUILIBRIUM)')
    axs[0, 0].grid(True)

    axs[0, 1].plot(distances, coordination_number, color='darkblue', lw=2)
    axs[0, 1].set_xlabel('Distance r (Å)')
    axs[0, 1].set_ylabel('Coordination Number N(r)')
    axs[0, 1].set_title('Integrated Total Coordination Number')
    axs[0, 1].grid(True)

    axs[1, 0].plot(distances, pmf_eq, color='crimson', lw=2)
    axs[1,0].set_xlim([0., 4.])
    axs[1, 0].set_xlabel('Distance r (Å)')
    axs[1, 0].set_ylabel('W(r) (in units of k_B T)')
    axs[1, 0].set_title('Potential of Mean Force (Total PMF)')
    axs[1, 0].grid(True)

    axs[1, 1].plot(time_steps_eq, g_max_eq, color='purple', marker='o', label='Peak Height (g_max)')
    axs[1, 1].set_xlabel('Window Center (Frame Index)')
    axs[1, 1].set_ylabel('g_max', color='purple')
    axs[1, 1].tick_params(axis='y', labelcolor='purple')

    ax2 = axs[1, 1].twinx()
    ax2.plot(time_steps_eq, fwhm_eq, color='orange', marker='s', label='Peak Width (FWHM)')
    ax2.set_ylabel('FWHM (Å)', color='orange')
    ax2.tick_params(axis='y', labelcolor='orange')
    axs[1, 1].set_title('Rolling Window Total Peak Tracking')
    axs[1, 1].grid(True)

    plt.tight_layout()

    # Save the figure instead of plotting it
    save_path = os.path.join(output_dir, "equilibrium_analysis_summary.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig) # Close the figure to free up memory


  ########## INDIVIDUAL PAIR ANALYSIS ############

  pairs = [
    ('Sn', 'I'),
    ('I', 'I'),
    ('N', 'I'),
    ('H', 'I'),
    ('C', 'N'),
    ('H', 'N'),
    ('H', 'C'),
  ]

  total_eq_frames = len(images_equilibrium)
  reference_atoms = images_equilibrium[0]
  volume = reference_atoms.get_volume()
  symbols = reference_atoms.get_chemical_symbols()
  partial_results_eq = {}

  def compute_pair_analysis(selected_elements):
    all_rdfs = []
    distances = None
    for atoms in images_equilibrium:
        rdf_values, distances = get_rdf(atoms, rmax=rmax, nbins=nbins, elements=selected_elements)
        all_rdfs.append(rdf_values)
    avg_rdf = np.mean(all_rdfs, axis=0)

    num_target_atoms = symbols.count(selected_elements[1])
    number_density = num_target_atoms / volume

    dr = distances[1] - distances[0]
    coordination_integrand = 4 * np.pi * number_density * (distances**2) * avg_rdf
    coordination_number = np.cumsum(coordination_integrand) * dr

    with np.errstate(divide='ignore'):
        pmf = -np.log(avg_rdf)
    pmf[np.isinf(pmf)] = np.nan

    time_steps = []
    g_max_history = []
    fwhm_history = []

    for start_idx in range(0, total_eq_frames - window_size + 1, window_size):
        window_images = images_equilibrium[start_idx : start_idx + window_size]
        window_rdfs = []
        for atoms in window_images:
            rdf_vals, _ = get_rdf(atoms, rmax=rmax, nbins=nbins, elements=selected_elements)
            window_rdfs.append(rdf_vals)
        window_avg_rdf = np.mean(window_rdfs, axis=0)

        max_idx = np.argmax(window_avg_rdf)
        g_max = window_avg_rdf[max_idx]
        half_max = g_max / 2.0

        left_indices = np.where(window_avg_rdf[:max_idx] < half_max)[0]
        right_indices = np.where(window_avg_rdf[max_idx:] < half_max)[0]

        if len(left_indices) > 0 and len(right_indices) > 0:
            r_left = distances[left_indices[-1]]
            r_right = distances[max_idx + right_indices[0]]
            fwhm = r_right - r_left
        else:
            fwhm = np.nan

        g_max_history.append(g_max)
        fwhm_history.append(fwhm)
        time_steps.append(start_idx + window_size // 2)

    return {
        "distances": distances,
        "avg_rdf": avg_rdf,
        "coordination_number": coordination_number,
        "pmf": pmf,
        "time_steps": np.array(time_steps),
        "g_max_history": np.array(g_max_history),
        "fwhm_history": np.array(fwhm_history),
    }

  if(Individual_pair==True):
    for pair in pairs:
      partial_results_eq[pair] = compute_pair_analysis(pair)
      print(f"Done computing for pair {pair}")
      pickle_path = os.path.join(output_dir, "partial_results_eq.pickle")
    with open(pickle_path, "wb") as f:
        pickle.dump(partial_results_eq, f)

    sections = {
      "Inorganic": [('Sn', 'I'), ('I', 'I')],
      "Organic vs Inorganic": [('N', 'I'), ('H', 'I')],
      "Organic": [('C', 'N'), ('H', 'N'), ('H', 'C')]
    }

    for section_name, pairs_in_section in sections.items():

        for pair in pairs_in_section:
            res = partial_results_eq[pair]
            distances = res["distances"]
            avg_rdf = res["avg_rdf"]
            coordination_number = res["coordination_number"]
            pmf = res["pmf"]
            time_steps = res["time_steps"]
            g_max_history = res["g_max_history"]
            fwhm_history = res["fwhm_history"]

            fig, axs = plt.subplots(2, 2, figsize=(14, 10))

            axs[0, 0].plot(distances, avg_rdf, color='darkgreen', lw=2)
            axs[0, 0].set_xlabel('Distance r (Å)')
            axs[0, 0].set_ylabel('g(r)')
            axs[0, 0].set_title(f'Standard Partial RDF ({pair[0]}-{pair[1]})')
            axs[0, 0].grid(True)

            axs[0, 1].plot(distances, coordination_number, color='darkblue', lw=2)
            axs[0, 1].set_xlabel('Distance r (Å)')
            axs[0, 1].set_ylabel('Coordination Number N(r)')
            axs[0, 1].set_title('Integrated Coordination Number')
            axs[0, 1].grid(True)

            axs[1, 0].plot(distances, pmf, color='crimson', lw=2)
            axs[1, 0].set_xlabel('Distance r (Å)')
            axs[1, 0].set_ylabel('W(r) (in units of k_B T)')
            axs[1, 0].set_title('Potential of Mean Force (PMF)')
            axs[1, 0].grid(True)

            axs[1, 1].plot(time_steps, g_max_history, color='purple', marker='o', label='Peak Height (g_max)')
            axs[1, 1].set_xlabel('Window Center (Frame Index)')
            axs[1, 1].set_ylabel('g_max', color='purple')
            axs[1, 1].tick_params(axis='y', labelcolor='purple')

            ax2 = axs[1, 1].twinx()
            ax2.plot(time_steps, fwhm_history, color='orange', marker='s', label='Peak Width (FWHM)')
            ax2.set_ylabel('FWHM (Å)', color='orange')
            ax2.tick_params(axis='y', labelcolor='orange')
            axs[1, 1].set_title('Rolling Window Peak Tracking')
            axs[1, 1].grid(True)

            fig.suptitle(f'{pair[0]}-{pair[1]} pair analysis (equilibrium) - {section_name}', fontsize=14)
            plt.tight_layout()

            # Save the figure dynamically using the atom pair names
            filename = f"partial_analysis_{pair[0]}_{pair[1]}.png"
            save_path = os.path.join(output_dir, filename)
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.close(fig) # Close the figure to free up memory


  if(ADF==True):
    ############### ADF I-Sn-I #####################

    center_element = 'Sn'
    neighbor_element = 'I'

    cutoff = 3.5 # similar as rmax before
    bins = 180
    total_counts = np.zeros(bins)
    bin_edges = np.linspace(0, 180, bins + 1)

    for atoms in images_equilibrium:
        frame_angles = []
        symbols = atoms.get_chemical_symbols()
        i, j, D = neighbor_list('ijD', atoms, cutoff=cutoff, self_interaction=False)

        for center_atom in range(len(atoms)):
            if symbols[center_atom] != center_element:
                continue

            mask = (i == center_atom)
            neighbor_indices = j[mask]
            vectors = D[mask]

            valid_mask = [symbols[idx] == neighbor_element for idx in neighbor_indices]
            valid_vectors = vectors[valid_mask]
            num_neighbors = len(valid_vectors)

            if num_neighbors >= 2:
                for idx1 in range(num_neighbors):
                    for idx2 in range(idx1 + 1, num_neighbors):
                        v1 = valid_vectors[idx1]
                        v2 = valid_vectors[idx2]
                        cos_theta = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
                        angle = np.degrees(np.arccos(np.clip(cos_theta, -1.0, 1.0)))
                        frame_angles.append(angle)

        if frame_angles:
            counts, _ = np.histogram(frame_angles, bins=bin_edges)
            total_counts += counts

    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    total_angles = np.sum(total_counts)

    if total_angles > 0:
        bin_width = bin_edges[1] - bin_edges[0]
        probability_density = total_counts / (total_angles * bin_width)
    else:
        probability_density = total_counts

    # Vertical lines
    plt.axvline(x=90, color='black', linestyle='--', linewidth=1, label='90 degs')
    plt.axvline(x=180, color='black', linestyle='--', linewidth=1)

    # Area ratio: find valley between peaks (restrict to 100-170° to avoid edges)
    valley_mask = (bin_centers >= 100) & (bin_centers <= 170)
    if np.any(valley_mask):
        valley_idx = np.argmin(probability_density[valley_mask]) + np.where(valley_mask)[0][0]
    else:
        valley_idx = len(bin_centers) // 2  # fallback
    plt.axvline(x=bin_centers[valley_idx], color='blue', linestyle='--', linewidth=1, label="Split to calc. ratio between two areas")

    area_left = np.sum(probability_density[:valley_idx]) * bin_width
    area_right = np.sum(probability_density[valley_idx:]) * bin_width
    ratio = area_left / area_right if area_right > 0 else np.inf

    print(f"Area (90° peak): {area_left:.4f}")
    print(f"Area (180° peak): {area_right:.4f}")

    plt.bar(bin_centers, probability_density, width=bin_width, color='crimson', alpha=0.7, edgecolor='none')
    plt.xlabel('Bond Angle (Degrees)')
    plt.ylabel('Probability Density (ADF)')
    plt.title(f'Partial ADF ({neighbor_element}-{center_element}-{neighbor_element})')
    plt.grid(True)
    plt.xlim(0, 180)
    plt.legend()

    # Save the figure instead of plotting it
    fig = plt.gcf() # Get the current figure since plt.figure() wasn't explicitly in this snippet
    filename = f"partial_adf_{neighbor_element}_{center_element}_{neighbor_element}.png"
    save_path = os.path.join(output_dir, filename)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig) # Close the figure to free up memory


    ############## ADF Sn-I-Sn ###############
    cutoff = 3.5
    center_element = 'I'
    neighbor_element = 'Sn'
    bins = 180
    total_counts = np.zeros(bins)
    bin_edges = np.linspace(0, 180, bins + 1)

    for atoms in images_equilibrium:
        frame_angles = []
        symbols = atoms.get_chemical_symbols()
        i, j, D = neighbor_list('ijD', atoms, cutoff=cutoff, self_interaction=False)

        for center_atom in range(len(atoms)):
            if symbols[center_atom] != center_element:
                continue

            mask = (i == center_atom)
            neighbor_indices = j[mask]
            vectors = D[mask]

            valid_mask = [symbols[idx] == neighbor_element for idx in neighbor_indices]
            valid_vectors = vectors[valid_mask]
            num_neighbors = len(valid_vectors)

            if num_neighbors >= 2:
                for idx1 in range(num_neighbors):
                    for idx2 in range(idx1 + 1, num_neighbors):
                        v1 = valid_vectors[idx1]
                        v2 = valid_vectors[idx2]
                        cos_theta = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
                        angle = np.degrees(np.arccos(np.clip(cos_theta, -1.0, 1.0)))
                        frame_angles.append(angle)

        if frame_angles:
            counts, _ = np.histogram(frame_angles, bins=bin_edges)
            total_counts += counts

    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    total_angles = np.sum(total_counts)

    if total_angles > 0:
        bin_width = bin_edges[1] - bin_edges[0]
        probability_density = total_counts / (total_angles * bin_width)
    else:
        probability_density = total_counts

    plt.bar(bin_centers, probability_density, width=bin_width, color='crimson', alpha=0.7, edgecolor='none')
    plt.xlabel('Bond Angle (Degrees)')
    plt.ylabel('Probability Density (ADF)')
    plt.title(f'Partial ADF ({neighbor_element}-{center_element}-{neighbor_element})')
    plt.grid(True)
    plt.xlim(0, 180)

    # Save the figure instead of plotting it
    fig = plt.gcf() # Get the current figure
    filename = f"partial_adf_{neighbor_element}_{center_element}_{neighbor_element}.png"
    save_path = os.path.join(output_dir, filename)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig) # Close the figure to free up memory

    ########### EVOLUTION OF PARTIAL ADF SN-I-SN #############

    center_element = 'I'
    neighbor_element = 'Sn'
    bins = 50
    bin_edges = np.linspace(0, 180, bins + 1)

    counts_early = np.zeros(bins)
    counts_mid = np.zeros(bins)
    counts_late = np.zeros(bins)

    third = total_frames // 3
    boundary_1 = third
    boundary_2 = 2 * third

    for idx, atoms in enumerate(images):
        frame_angles = []
        symbols = atoms.get_chemical_symbols()
        i, j, D = neighbor_list('ijD', atoms, cutoff=cutoff, self_interaction=False)

        for center_atom in range(len(atoms)):
            if symbols[center_atom] != center_element:
                continue

            mask = (i == center_atom)
            neighbor_indices = j[mask]
            vectors = D[mask]

            valid_mask = [symbols[idx_n] == neighbor_element for idx_n in neighbor_indices]
            valid_vectors = vectors[valid_mask]
            num_neighbors = len(valid_vectors)

            if num_neighbors >= 2:
                for idx1 in range(num_neighbors):
                    for idx2 in range(idx1 + 1, num_neighbors):
                        v1 = valid_vectors[idx1]
                        v2 = valid_vectors[idx2]
                        cos_theta = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
                        angle = np.degrees(np.arccos(np.clip(cos_theta, -1.0, 1.0)))
                        frame_angles.append(angle)

        if frame_angles:
            counts, _ = np.histogram(frame_angles, bins=bin_edges)
            if idx < boundary_1:
                counts_early += counts
            elif idx < boundary_2:
                counts_mid += counts
            else:
                counts_late += counts

    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    bin_width = bin_edges[1] - bin_edges[0]

    sum_early = np.sum(counts_early)
    sum_mid = np.sum(counts_mid)
    sum_late = np.sum(counts_late)

    pdf_early = counts_early / (sum_early * bin_width) if sum_early > 0 else counts_early
    pdf_mid = counts_mid / (sum_mid * bin_width) if sum_mid > 0 else counts_mid
    pdf_late = counts_late / (sum_late * bin_width) if sum_late > 0 else counts_late
    fig = plt.figure(figsize=(10, 6))
    plt.plot(bin_centers, pdf_early, color='royalblue', lw=2, label='Early Stage (Beginning)')
    plt.plot(bin_centers, pdf_mid, color='orange', lw=2, label='Middle Stage (Relaxation)')
    plt.plot(bin_centers, pdf_late, color='crimson', lw=2, label='Late Stage (Equilibrium)')

    plt.xlabel('Bond Angle (Degrees)')
    plt.ylabel('Probability Density (ADF)')
    plt.title(f'Evolution of Partial ADF ({neighbor_element}-{center_element}-{neighbor_element})')
    plt.grid(True)
    plt.xlim(0, 180)
    plt.legend()
    plt.tight_layout()

    # Save the figure instead of plotting it
    filename = f"adf_evolution_{neighbor_element}_{center_element}_{neighbor_element}.png"
    save_path = os.path.join(output_dir, filename)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig) # Close the figure to free up memory


    ############# H-C-H ANALYSIS ##############
    cutoff = 1.5
    center_element = 'C'
    neighbor_element = 'H'
    bins = 180
    total_counts = np.zeros(bins)
    bin_edges = np.linspace(0, 180, bins + 1)

    for atoms in images_equilibrium:
        frame_angles = []
        symbols = atoms.get_chemical_symbols()
        i, j, D = neighbor_list('ijD', atoms, cutoff=cutoff, self_interaction=False)

        for center_atom in range(len(atoms)):
            if symbols[center_atom] != center_element:
                continue

            mask = (i == center_atom)
            neighbor_indices = j[mask]
            vectors = D[mask]

            valid_mask = [symbols[idx] == neighbor_element for idx in neighbor_indices]
            valid_vectors = vectors[valid_mask]
            num_neighbors = len(valid_vectors)

            if num_neighbors >= 2:
                for idx1 in range(num_neighbors):
                    for idx2 in range(idx1 + 1, num_neighbors):
                        v1 = valid_vectors[idx1]
                        v2 = valid_vectors[idx2]
                        cos_theta = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
                        angle = np.degrees(np.arccos(np.clip(cos_theta, -1.0, 1.0)))
                        frame_angles.append(angle)

        if frame_angles:
            counts, _ = np.histogram(frame_angles, bins=bin_edges)
            total_counts += counts

    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    total_angles = np.sum(total_counts)

    if total_angles > 0:
        bin_width = bin_edges[1] - bin_edges[0]
        probability_density = total_counts / (total_angles * bin_width)
    else:
        probability_density = total_counts
    plt.bar(bin_centers, probability_density, width=bin_width, color='crimson', alpha=0.7, edgecolor='none')
    plt.xlabel('Bond Angle (Degrees)')
    plt.ylabel('Probability Density (ADF)')
    plt.title(f'Partial ADF ({neighbor_element}-{center_element}-{neighbor_element})')
    plt.grid(True)
    plt.xlim(0, 180)

    # Save the figure instead of plotting it
    fig = plt.gcf() # Get the current figure
    filename = f"partial_adf_{neighbor_element}_{center_element}_{neighbor_element}.png"
    save_path = os.path.join(output_dir, filename)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig) # Close the figure to free up memory

    ############ EVOLUTION OF HCH ADF #########
    cutoff = 1.5
    center_element = 'C'
    neighbor_element = 'H'
    bins = 50
    bin_edges = np.linspace(0, 180, bins + 1)

    counts_early = np.zeros(bins)
    counts_mid = np.zeros(bins)
    counts_late = np.zeros(bins)

    third = total_frames // 3
    boundary_1 = third
    boundary_2 = 2 * third

    for idx, atoms in enumerate(images):
        frame_angles = []
        symbols = atoms.get_chemical_symbols()
        i, j, D = neighbor_list('ijD', atoms, cutoff=cutoff, self_interaction=False)

        for center_atom in range(len(atoms)):
            if symbols[center_atom] != center_element:
                continue

            mask = (i == center_atom)
            neighbor_indices = j[mask]
            vectors = D[mask]

            valid_mask = [symbols[idx_n] == neighbor_element for idx_n in neighbor_indices]
            valid_vectors = vectors[valid_mask]
            num_neighbors = len(valid_vectors)

            if num_neighbors >= 2:
                for idx1 in range(num_neighbors):
                    for idx2 in range(idx1 + 1, num_neighbors):
                        v1 = valid_vectors[idx1]
                        v2 = valid_vectors[idx2]
                        cos_theta = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
                        angle = np.degrees(np.arccos(np.clip(cos_theta, -1.0, 1.0)))
                        frame_angles.append(angle)

        if frame_angles:
            counts, _ = np.histogram(frame_angles, bins=bin_edges)
            if idx < boundary_1:
                counts_early += counts
            elif idx < boundary_2:
                counts_mid += counts
            else:
                counts_late += counts

    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    bin_width = bin_edges[1] - bin_edges[0]

    sum_early = np.sum(counts_early)
    sum_mid = np.sum(counts_mid)
    sum_late = np.sum(counts_late)

    pdf_early = counts_early / (sum_early * bin_width) if sum_early > 0 else counts_early
    pdf_mid = counts_mid / (sum_mid * bin_width) if sum_mid > 0 else counts_mid
    pdf_late = counts_late / (sum_late * bin_width) if sum_late > 0 else counts_late
    fig = plt.figure(figsize=(10, 6))
    plt.plot(bin_centers, pdf_early, color='royalblue', lw=2, label='Early Stage (Beginning)')
    plt.plot(bin_centers, pdf_mid, color='orange', lw=2, label='Middle Stage (Relaxation)')
    plt.plot(bin_centers, pdf_late, color='crimson', lw=2, label='Late Stage (Equilibrium)')

    plt.xlabel('Bond Angle (Degrees)')
    plt.ylabel('Probability Density (ADF)')
    plt.title(f'Evolution of Partial ADF ({neighbor_element}-{center_element}-{neighbor_element})')
    plt.grid(True)
    plt.xlim(0, 180)
    plt.legend()
    plt.tight_layout()

    # Save the figure instead of plotting it
    # Added '_stages' to the filename to ensure it doesn't overwrite earlier plots
    filename = f"adf_evolution_stages_{neighbor_element}_{center_element}_{neighbor_element}.png"
    save_path = os.path.join(output_dir, filename)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig) # Close the figure to free up memory




    ############# H-N-H ANALYSIS ##############

    cutoff = 1.5
    center_element = 'N'
    neighbor_element = 'H'
    bins = 180
    total_counts = np.zeros(bins)
    bin_edges = np.linspace(0, 180, bins + 1)

    for atoms in images_equilibrium:
        frame_angles = []
        symbols = atoms.get_chemical_symbols()
        i, j, D = neighbor_list('ijD', atoms, cutoff=cutoff, self_interaction=False)

        for center_atom in range(len(atoms)):
            if symbols[center_atom] != center_element:
                continue

            mask = (i == center_atom)
            neighbor_indices = j[mask]
            vectors = D[mask]

            valid_mask = [symbols[idx] == neighbor_element for idx in neighbor_indices]
            valid_vectors = vectors[valid_mask]
            num_neighbors = len(valid_vectors)

            if num_neighbors >= 2:
                for idx1 in range(num_neighbors):
                    for idx2 in range(idx1 + 1, num_neighbors):
                        v1 = valid_vectors[idx1]
                        v2 = valid_vectors[idx2]
                        cos_theta = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
                        angle = np.degrees(np.arccos(np.clip(cos_theta, -1.0, 1.0)))
                        frame_angles.append(angle)

        if frame_angles:
            counts, _ = np.histogram(frame_angles, bins=bin_edges)
            total_counts += counts

    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    total_angles = np.sum(total_counts)

    if total_angles > 0:
        bin_width = bin_edges[1] - bin_edges[0]
        probability_density = total_counts / (total_angles * bin_width)
    else:
      probability_density = total_counts
    plt.bar(bin_centers, probability_density, width=bin_width, color='crimson', alpha=0.7, edgecolor='none')
    plt.xlabel('Bond Angle (Degrees)')
    plt.ylabel('Probability Density (ADF)')
    plt.title(f'Partial ADF ({neighbor_element}-{center_element}-{neighbor_element})')
    plt.grid(True)
    plt.xlim(0, 180)

    # Save the figure instead of plotting it
    fig = plt.gcf() # Get the current figure
    # Added an extra identifier to the filename to avoid overwriting previous plots
    filename = f"partial_adf_bar_{neighbor_element}_{center_element}_{neighbor_element}.png"
    save_path = os.path.join(output_dir, filename)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig) # Close the figure to free up memory

    ############  EVOLUTION OF HNH ADF ###########
    cutoff = 1.5
    center_element = 'N'
    neighbor_element = 'H'
    bins = 50
    bin_edges = np.linspace(0, 180, bins + 1)

    counts_early = np.zeros(bins)
    counts_mid = np.zeros(bins)
    counts_late = np.zeros(bins)

    third = total_frames // 3
    boundary_1 = third
    boundary_2 = 2 * third

    for idx, atoms in enumerate(images):
        frame_angles = []
        symbols = atoms.get_chemical_symbols()
        i, j, D = neighbor_list('ijD', atoms, cutoff=cutoff, self_interaction=False)

        for center_atom in range(len(atoms)):
            if symbols[center_atom] != center_element:
                continue

            mask = (i == center_atom)
            neighbor_indices = j[mask]
            vectors = D[mask]

            valid_mask = [symbols[idx_n] == neighbor_element for idx_n in neighbor_indices]
            valid_vectors = vectors[valid_mask]
            num_neighbors = len(valid_vectors)

            if num_neighbors >= 2:
                for idx1 in range(num_neighbors):
                    for idx2 in range(idx1 + 1, num_neighbors):
                        v1 = valid_vectors[idx1]
                        v2 = valid_vectors[idx2]
                        cos_theta = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
                        angle = np.degrees(np.arccos(np.clip(cos_theta, -1.0, 1.0)))
                        frame_angles.append(angle)

        if frame_angles:
            counts, _ = np.histogram(frame_angles, bins=bin_edges)
            if idx < boundary_1:
                counts_early += counts
            elif idx < boundary_2:
                counts_mid += counts
            else:
                counts_late += counts

    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    bin_width = bin_edges[1] - bin_edges[0]

    sum_early = np.sum(counts_early)
    sum_mid = np.sum(counts_mid)
    sum_late = np.sum(counts_late)

    pdf_early = counts_early / (sum_early * bin_width) if sum_early > 0 else counts_early
    pdf_mid = counts_mid / (sum_mid * bin_width) if sum_mid > 0 else counts_mid
    pdf_late = counts_late / (sum_late * bin_width) if sum_late > 0 else counts_late
    fig = plt.figure(figsize=(10, 6))
    plt.plot(bin_centers, pdf_early, color='royalblue', lw=2, label='Early Stage (Beginning)')
    plt.plot(bin_centers, pdf_mid, color='orange', lw=2, label='Middle Stage (Relaxation)')
    plt.plot(bin_centers, pdf_late, color='crimson', lw=2, label='Late Stage (Equilibrium)')

    plt.xlabel('Bond Angle (Degrees)')
    plt.ylabel('Probability Density (ADF)')
    plt.title(f'Evolution of Partial ADF ({neighbor_element}-{center_element}-{neighbor_element})')
    plt.grid(True)
    plt.xlim(0, 180)
    plt.legend()
    plt.tight_layout()

    # Save the figure instead of plotting it
    # Added '_v2' to the filename to avoid overwriting the previous identical plot
    filename = f"adf_evolution_stages_v2_{neighbor_element}_{center_element}_{neighbor_element}.png"
    save_path = os.path.join(output_dir, filename)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig) # Close the figure to free up memory


  if(OACF==True):
    ############ OACF BASED ANALYSIS SN-I #############
    num_frames = len(images)

    first_frame = images[0]
    symbols = first_frame.get_chemical_symbols()
    i, j = neighbor_list('ij', first_frame, cutoff=3.9, self_interaction=False)

    pairs = []
    for idx_i, idx_j in zip(i, j):
        if symbols[idx_i] == 'I' and symbols[idx_j] == 'Sn':
            pairs.append((idx_i, idx_j))

    num_molecules = len(pairs)
    vectors = np.zeros((num_frames, num_molecules, 3))

    for f_idx, atoms in enumerate(images):
        for m_idx, (c_idx, n_idx) in enumerate(pairs):
            vec = atoms.get_distance(c_idx, n_idx, vector=True, mic=True)
            vectors[f_idx, m_idx] = vec / np.linalg.norm(vec)

    oacf = np.zeros(num_frames)
    for delta in range(num_frames):
        v_t = vectors[:num_frames - delta]
        v_tdelta = vectors[delta:]
        oacf[delta] = np.mean(np.sum(v_t * v_tdelta, axis=2))

    time_axis = np.arange(num_frames)
    fig = plt.figure(figsize=(8, 5))
    plt.plot(time_axis, oacf, color='darkviolet', lw=2)
    plt.xlabel('Time Lag (Frames)')
    plt.ylabel('C(t)')
    plt.title('Orientational Autocorrelation Function (OACF) of Sn-I Vectors')
    plt.grid(True)
    plt.ylim(0.9, 1.)
    plt.tight_layout()

    # Save the figure instead of plotting it
    save_path = os.path.join(output_dir, "oacf_sn_i_vectors.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig) # Close the figure to free up memory


    ############# OACF BASED ANALYSIS C-N   ################

    first_frame = images[0]
    symbols = first_frame.get_chemical_symbols()
    i, j = neighbor_list('ij', first_frame, cutoff=1.6, self_interaction=False)

    pairs = []
    for idx_i, idx_j in zip(i, j):
        if symbols[idx_i] == 'C' and symbols[idx_j] == 'H':
            pairs.append((idx_i, idx_j))

    num_molecules = len(pairs)
    vectors = np.zeros((num_frames, num_molecules, 3))

    for f_idx, atoms in enumerate(images):
        for m_idx, (c_idx, n_idx) in enumerate(pairs):
            vec = atoms.get_distance(c_idx, n_idx, vector=True, mic=True)
            vectors[f_idx, m_idx] = vec / np.linalg.norm(vec)

    oacf = np.zeros(num_frames)
    for delta in range(num_frames):
        v_t = vectors[:num_frames - delta]
        v_tdelta = vectors[delta:]
        oacf[delta] = np.mean(np.sum(v_t * v_tdelta, axis=2))

    time_axis = np.arange(num_frames)

    fig = plt.figure(figsize=(8, 5))
    plt.plot(time_axis, oacf, color='darkviolet', lw=2)
    plt.xlabel('Time Lag (Frames)')
    plt.ylabel('C(t)')
    plt.title('Orientational Autocorrelation Function (OACF) of C-H Vectors')
    plt.grid(True)
    plt.ylim(-0.1, 1.1)
    plt.tight_layout()

    # Save the figure instead of plotting it
    save_path = os.path.join(output_dir, "oacf_c_h_vectors.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig) # Close the figure to free up memory




    ############# OACF BASED ANALYSIS N-H   ################

    first_frame = images[0]
    symbols = first_frame.get_chemical_symbols()
    i, j = neighbor_list('ij', first_frame, cutoff=1.6, self_interaction=False)

    pairs = []
    for idx_i, idx_j in zip(i, j):
        if symbols[idx_i] == 'H' and symbols[idx_j] == 'N':
            pairs.append((idx_i, idx_j))

    num_molecules = len(pairs)
    vectors = np.zeros((num_frames, num_molecules, 3))

    for f_idx, atoms in enumerate(images):
        for m_idx, (c_idx, n_idx) in enumerate(pairs):
            vec = atoms.get_distance(c_idx, n_idx, vector=True, mic=True)
            vectors[f_idx, m_idx] = vec / np.linalg.norm(vec)

    oacf = np.zeros(num_frames)
    for delta in range(num_frames):
        v_t = vectors[:num_frames - delta]
        v_tdelta = vectors[delta:]
        oacf[delta] = np.mean(np.sum(v_t * v_tdelta, axis=2))

    time_axis = np.arange(num_frames)

    fig = plt.figure(figsize=(8, 5))
    plt.plot(time_axis, oacf, color='darkviolet', lw=2)
    plt.xlabel('Time Lag (Frames)')
    plt.ylabel('C(t)')
    plt.title('Orientational Autocorrelation Function (OACF) of H-N Vectors')
    plt.grid(True)
    plt.ylim(-0.1, 1.1)
    plt.tight_layout()

    # Save the figure instead of plotting it
    save_path = os.path.join(output_dir, "oacf_h_n_vectors.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig) # Close the figure to free up memory



  if(VDOS==True):
    ############ PARTIAL VDOS FOR ALL ELEMENTS ############

    timestep = 0.5

    first_frame = images_equilibrium[0]
    symbols = np.array(first_frame.get_chemical_symbols())
    unique_elements = sorted(list(set(symbols)))

    cell = first_frame.get_cell()
    pbc = first_frame.get_pbc()

    all_velocities = []
    prev_pos = images_equilibrium[0].get_positions()
    for atoms in images_equilibrium[1:]:
        pos = atoms.get_positions()
        step_disp = pos - prev_pos
        step_disp, _ = find_mic(step_disp, cell, pbc=pbc)
        all_velocities.append(step_disp / timestep)
        prev_pos = pos

    all_velocities = np.array(all_velocities)
    num_frames = len(all_velocities)
    frequencies = np.fft.rfftfreq(num_frames, d=timestep)

    print("Max v dataset:", np.max(np.abs(all_velocities)))

    vdos_data = {}
    total_vdos = np.zeros(len(frequencies))

    for element in unique_elements:
        mask = (symbols == element)
        element_velocities = all_velocities[:, mask, :]

        fft_v = np.fft.rfft(element_velocities, axis=0)
        power_spectrum = np.abs(fft_v) ** 2
        pvdos = np.sum(power_spectrum, axis=(1, 2))

        vdos_data[element] = pvdos
        total_vdos += pvdos

    n_elements = len(unique_elements)
    fig, axes = plt.subplots(n_elements, 1, figsize=(10, 4*n_elements), sharex=True)
    if n_elements == 1:
        axes = [axes]

    for ax, (element, pvdos) in zip(axes, vdos_data.items()):
        ax.plot(frequencies, pvdos, label=f'{element} pVDOS', lw=1.5)
        ax.plot(frequencies, total_vdos, label='Total VDOS', color='black', lw=2, linestyle='--')
        ax.set_ylabel('VDOS (a.u.)')
        ax.set_title(f'{element} Partial VDOS')
        ax.set_xlim(frequencies[0], 0.1)
        ax.grid(True)
        ax.legend()

    axes[-1].set_xlabel('Frequency (10¹⁵ Hz)')
    plt.tight_layout()

    # Save the figure instead of plotting it
    save_path = os.path.join(output_dir, "vdos_partial_elements.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig) # Close the figure to free up memory




    ####### VDOS LOW FREQ , HIGH FREQ AND H ONLY ###########

    timestep = 0.5

    first_frame = images_equilibrium[0]
    symbols = np.array(first_frame.get_chemical_symbols())
    masses = first_frame.get_masses()
    unique_elements = sorted(list(set(symbols)))

    cell = first_frame.get_cell()
    pbc = first_frame.get_pbc()

    all_velocities = []
    prev_pos = images_equilibrium[0].get_positions()
    for atoms in images_equilibrium[1:]:
        pos = atoms.get_positions()
        step_disp = pos - prev_pos
        step_disp, _ = find_mic(step_disp, cell, pbc=pbc)
        all_velocities.append(step_disp / timestep)
        prev_pos = pos

    all_velocities = np.array(all_velocities)
    num_frames = len(all_velocities)
    frequencies = np.fft.rfftfreq(num_frames, d=timestep)

    vdos_data = {}
    total_vdos = np.zeros(len(frequencies))

    for element in unique_elements:
        mask = (symbols == element)
        element_velocities = all_velocities[:, mask, :]
        element_mass = masses[mask][0]

        fft_v = np.fft.rfft(element_velocities, axis=0)
        power_spectrum = np.abs(fft_v) ** 2
        pvdos = np.sum(power_spectrum, axis=(1, 2)) * element_mass

        vdos_data[element] = pvdos
        total_vdos += pvdos

    fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True)
    colors = {'C': 'blue', 'N': 'red', 'Sn': 'green', 'I': 'purple'}

    # Top: C, N
    for element in ['C', 'N']:
        if element in vdos_data:
            ax1.plot(frequencies, vdos_data[element], label=f'{element}', lw=1.5, color=colors[element])
    ax1.set_ylabel('VDOS (a.u.)')
    ax1.set_title('"Low" Frequency (No H)')
    ax1.legend()
    ax1.grid(True)

    # Bottom: Sn, I
    for element in ['Sn', 'I']:
        if element in vdos_data:
            ax2.plot(frequencies, vdos_data[element], label=f'{element}', lw=1.5, color=colors[element])
    ax2.set_xlabel('Frequency (10¹⁵ Hz)')
    ax2.set_ylabel('VDOS (a.u.)')
    ax2.legend()
    ax2.grid(True)

    ax2.set_xlim([0., 0.01])
    plt.tight_layout()

    # Save the figure instead of plotting it
    save_path = os.path.join(output_dir, "vdos_low_frequency.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig) # Close the figure to free up memory

    low_freq_vdos_analysis = {
      "frequencies": frequencies,
      "vdos_data": {k: v for k, v in vdos_data.items() if k in ['C', 'N', 'Sn', 'I']},
      "colors": colors,
    }
    pickle_path = os.path.join(output_dir, "low_freq_vdos_analysis.pickle")
    with open(pickle_path, "wb") as f:
        pickle.dump(low_freq_vdos_analysis, f)

    # --- Plot total VDOS in a specific (visually identified) frequency range ---
    fig = plt.figure(figsize=(10, 6))

    # Plot total VDOS
    plt.plot(frequencies, total_vdos, color='black', lw=2, label='Total VDOS')

    # Set x-axis limits to the desired range (0.05 to 0.07 in 10^15 Hz)
    plt.xlim(0.05, 0.07)
    plt.ylim([0., 1250])

    plt.xlabel('Frequency (10¹⁵ Hz)')
    plt.ylabel('VDOS (a.u.)')
    plt.title('Total VDOS - "high" frequency of interest')
    plt.grid(True, alpha=0.3)
    plt.legend()

    plt.tight_layout()

    # Save the figure instead of plotting it
    save_path = os.path.join(output_dir, "vdos_high_frequency_range.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig) # Close the figure to free up memory


    fig = plt.figure(figsize=(14,6))
    if 'H' in vdos_data:
        plt.plot(frequencies, vdos_data['H'], label='H', lw=1.5, color='black')

    plt.xlim([0.,0.07])
    plt.xlabel('Frequency (10^15 Hz)')
    plt.ylabel('VDOS (a.u.)')
    plt.title('VDOS for Hydrogen Only')
    plt.legend()
    plt.grid(True)

    # Save the figure instead of plotting it
    save_path = os.path.join(output_dir, "vdos_hydrogen_only.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig) # Close the figure to free up memory

    hydrogen_vdos_analysis = {
      "frequencies": frequencies,
      "vdos_H": vdos_data.get('H', None),
    }
    pickle_path = os.path.join(output_dir, "hydrogen_vdos_analysis.pickle")
    with open(pickle_path, "wb") as f:
        pickle.dump(hydrogen_vdos_analysis, f)



  ORGANIC_SYMBOLS = {"H", "C", "N"}
  INORGANIC_SYMBOLS = {"Sn", "I"}

  input_file = input_traj_dir
  if(CompleteMSD==True):
    ############ COMPLETE MSD ANALYSIS ###########
    ORGANIC_SYMBOLS = {"H", "C", "N"}
    INORGANIC_SYMBOLS = {"Sn", "I"}

    input_file = input_traj_dir

    if not os.path.exists(input_file):
        print(f"Error: {input_file} not found.")
        exit(1)

    times = []
    msd_full = []
    msd_org = []
    msd_inorg = []
    msd_h = []

    rmsd_full = []
    rmsd_org = []
    rmsd_inorg = []
    rmsd_h = []

    ref_pos_full = None
    ref_pos_org = None
    ref_pos_inorg = None
    ref_pos_h = None

    for index, atoms in enumerate(iread(input_file, index=":")):
        times.append(index)

        #splitting indexes
        org_indices = [atom.index for atom in atoms if atom.symbol in ORGANIC_SYMBOLS]
        inorg_indices = [atom.index for atom in atoms if atom.symbol in INORGANIC_SYMBOLS]
        h_indices = [atom.index for atom in atoms if atom.symbol == "H"]

        #taking position
        pos_full = atoms.get_positions()
        pos_org = pos_full[org_indices] if org_indices else np.array([])
        pos_inorg = pos_full[inorg_indices] if inorg_indices else np.array([])
        pos_h = pos_full[h_indices] if h_indices else np.array([])

        #starting point
        if index == 0:
            ref_pos_full = pos_full.copy()
            ref_pos_org = pos_org.copy() if pos_org.size > 0 else None
            ref_pos_inorg = pos_inorg.copy() if pos_inorg.size > 0 else None
            ref_pos_h = pos_h.copy() if pos_h.size > 0 else None

    # unwrapped position accumulators, initialized at frame 0
        if index == 0:
            prev_pos_full = pos_full.copy()
            unwrapped_pos_full = pos_full.copy()
        else:
            step_disp = pos_full - prev_pos_full
            step_disp, _ = find_mic(step_disp, atoms.get_cell(), pbc=atoms.get_pbc())
            unwrapped_pos_full += step_disp
            prev_pos_full = pos_full.copy()
        unwrapped_pos_org = unwrapped_pos_full[org_indices] if org_indices else np.array([])
        unwrapped_pos_inorg = unwrapped_pos_full[inorg_indices] if inorg_indices else np.array([])
        unwrapped_pos_h = unwrapped_pos_full[h_indices] if h_indices else np.array([])

        sq_dist_full = np.sum((unwrapped_pos_full - ref_pos_full) ** 2, axis=1)

        msd_full.append(np.mean(sq_dist_full))

        if ref_pos_org is not None and pos_org.size > 0:
            sq_dist_org = np.sum((unwrapped_pos_org - ref_pos_org) ** 2, axis=1)
            msd_org.append(np.mean(sq_dist_org))
        else:
            msd_org.append(0.0)

        if ref_pos_inorg is not None and pos_inorg.size > 0:
            sq_dist_inorg = np.sum((unwrapped_pos_inorg - ref_pos_inorg) ** 2, axis=1)
            msd_inorg.append(np.mean(sq_dist_inorg))
        else:
            msd_inorg.append(0.0)

        if ref_pos_h is not None and pos_h.size > 0:
            sq_dist_h = np.sum((unwrapped_pos_h - ref_pos_h) ** 2, axis=1)
            msd_h.append(np.mean(sq_dist_h))
        else:
            msd_h.append(0.0)

        #root mean squared deviation
        rmsd_full.append(np.sqrt(np.mean(sq_dist_full)))

        if ref_pos_org is not None and pos_org.size > 0:
            rmsd_org.append(np.sqrt(np.mean(sq_dist_org)))
        else:
            rmsd_org.append(0.0)

        if ref_pos_inorg is not None and pos_inorg.size > 0:
            rmsd_inorg.append(np.sqrt(np.mean(sq_dist_inorg)))
        else:
            rmsd_inorg.append(0.0)

        if ref_pos_h is not None and pos_h.size > 0:
            rmsd_h.append(np.sqrt(np.mean(sq_dist_h)))
        else:
            rmsd_h.append(0.0)

    print(f"Processed {len(times)} frames successfully.")

    #saving result
    times = np.array(times)
    msd_full = np.array(msd_full)
    msd_org = np.array(msd_org)
    msd_inorg = np.array(msd_inorg)
    msd_h = np.array(msd_h)

    rmsd_full = np.array(rmsd_full)
    rmsd_org = np.array(rmsd_org)
    rmsd_inorg = np.array(rmsd_inorg)
    rmsd_h = np.array(rmsd_h)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

    # Top Plot: MSD
    ax1.plot(times, msd_full, label="Full Structure", color="black", linewidth=2)
    ax1.plot(times, msd_org, label="Organic Sublattice (H, C, N)", color="brown", linestyle="--")
    ax1.plot(times, msd_inorg, label="Inorganic Sublattice (Sn, I)", color="purple", linestyle="-.")
    ax1.plot(times, msd_h, label="Hydrogen Only (H)", color="darkorange", linestyle=":")
    ax1.set_ylabel("Mean Squared Displacement (Å²)")
    ax1.set_title("Trajectory Dynamics Comparison: Organic vs Inorganic Parts & Hydrogen")
    ax1.legend()
    ax1.grid(True, linestyle=":", alpha=0.6)

    # Bottom Plot: RMSD
    ax2.plot(times, rmsd_full, label="Full Structure", color="black", linewidth=2)
    ax2.plot(times, rmsd_org, label="Organic Sublattice", color="brown", linestyle="--")
    ax2.plot(times, rmsd_inorg, label="Inorganic Sublattice", color="purple", linestyle="-.")
    ax2.plot(times, rmsd_h, label="Hydrogen Only", color="darkorange", linestyle=":")
    ax2.set_ylabel("RMSD (Å)")
    ax2.set_xlabel("Frame Index / Time step")
    ax2.legend()
    ax2.grid(True, linestyle=":", alpha=0.6)

    plt.tight_layout()

    # Save the figure instead of plotting it
    save_path = os.path.join(output_dir, "trajectory_dynamics_comparison.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig) # Close the figure to free up memory
    msd_rmsd_analysis = {
      "times": times,
      "msd_full": msd_full,
      "msd_org": msd_org,
      "msd_inorg": msd_inorg,
      "msd_h": msd_h,
      "rmsd_full": rmsd_full,
      "rmsd_org": rmsd_org,
      "rmsd_inorg": rmsd_inorg,
      "rmsd_h": rmsd_h,
    }
    pickle_path = os.path.join(output_dir, "msd_rmsd_analysis.pickle")
    with open(pickle_path, "wb") as f:
        pickle.dump(msd_rmsd_analysis, f)

    # ------------------------------------------------------------
    # 2. Initialise dictionaries
    # ------------------------------------------------------------
    times = []
    msd_elements = {}
    rmsd_elements = {}
    ref_positions = {}          # initial unwrapped positions per element
    prev_pos_full = None        # for frame‑to‑frame displacement
    unwrapped_pos_full = None   # cumulative unwrapped positions

    # ------------------------------------------------------------
    # 3. Loop over frames
    # ------------------------------------------------------------
    for index, atoms in enumerate(iread(input_file, index=":")):
        times.append(index)

        # Current wrapped positions and symbols
        pos_full = atoms.get_positions()
        symbols = np.array(atoms.get_chemical_symbols())
        unique_elements = sorted(set(symbols))

        # ---- First frame: set up references ----
        if index == 0:
            # Initialise dictionaries for all elements
            for element in unique_elements:
                msd_elements[element] = []
                rmsd_elements[element] = []
            # Set reference positions (will be overwritten after unwrapping)
            # We will set them after unwrapping the first frame

            # Initialise unwrapping
            prev_pos_full = pos_full.copy()
            unwrapped_pos_full = pos_full.copy()   # at frame 0, unwrapped = wrapped

            # Now store reference positions for each element from the unwrapped frame 0
            for element in unique_elements:
                mask = (symbols == element)
                ref_positions[element] = unwrapped_pos_full[mask].copy()

        else:
            # ---- Compute frame‑to‑frame displacement with MIC ----
            step_disp = pos_full - prev_pos_full
            step_disp, _ = find_mic(step_disp, atoms.get_cell(), pbc=atoms.get_pbc())
            unwrapped_pos_full += step_disp
            prev_pos_full = pos_full.copy()

        # ---- For each element, compute MSD/RMSD from unwrapped positions ----
        for element in unique_elements:
            mask = (symbols == element)
            if np.any(mask):
                # Unwrapped positions for this element
                unwrapped_pos_element = unwrapped_pos_full[mask]
                # Reference positions (initial unwrapped)
                ref_pos_element = ref_positions[element]

                # Squared displacements
                sq_dist = np.sum((unwrapped_pos_element - ref_pos_element) ** 2, axis=1)
                mean_sq_dist = np.mean(sq_dist)

                msd_elements[element].append(mean_sq_dist)
                rmsd_elements[element].append(np.sqrt(mean_sq_dist))
            else:
                # Should not happen because we only loop over elements present
                msd_elements[element].append(0.0)
                rmsd_elements[element].append(0.0)

    # ------------------------------------------------------------
    # 4. Convert to numpy arrays for plotting
    # ------------------------------------------------------------
    times = np.array(times)
    for element in msd_elements:
        msd_elements[element] = np.array(msd_elements[element])
        rmsd_elements[element] = np.array(rmsd_elements[element])

    print(f"Processed {len(times)} frames successfully. Elements: {list(msd_elements.keys())}")
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

    for element in msd_elements.keys():
        ax1.plot(times, msd_elements[element], label=f"Element {element}", linewidth=1.5)
        ax2.plot(times, rmsd_elements[element], label=f"Element {element}", linewidth=1.5)

    ax1.set_ylabel("Mean Squared Displacement (Å²)")
    ax1.set_title("MSD Comparison Between All Chemical Species")
    ax1.legend()
    ax1.grid(True, linestyle=":", alpha=0.6)

    ax2.set_ylabel("RMSD (Å)")
    ax2.set_xlabel("Frame Index / Time step")
    ax2.set_title("RMSD Comparison Between All Chemical Species")
    ax2.legend()
    ax2.grid(True, linestyle=":", alpha=0.6)

    plt.tight_layout()

    # Save the figure instead of plotting it
    save_path = os.path.join(output_dir, "element_dynamics_comparison.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig) # Close the figure to free up memory


  timestep_in_fs = 0.5

  all_velocities = []
  if(TemperatureCrossCheck==True):
    ############# TEMPERATURE CROSS CHECK ##################
    timestep_in_fs = 0.5

    all_velocities = []
    for i in range(len(images_equilibrium) - 1):
        delta_p = images_equilibrium[i+1].get_positions() - images_equilibrium[i].get_positions()
        delta_p, _ = find_mic(delta_p, images_equilibrium[i].get_cell(), pbc=images_equilibrium[i].get_pbc())
        all_velocities.append(delta_p / timestep_in_fs)

    all_velocities = np.array(all_velocities)
    N_atoms = len(images_equilibrium[0])
    dynamic_dof = 3 * N_atoms - 3
    temperatures = []

    for i, atoms in enumerate(images_equilibrium[:-1]):
        masses = atoms.get_masses()
        v_ase = all_velocities[i] / units.fs
        ek = 0.5 * np.sum(masses[:, np.newaxis] * (v_ase ** 2))
        t_instantaneous = (2 * ek) / (dynamic_dof * units.kB)
        temperatures.append(t_instantaneous)

    mean_temperature = np.mean(temperatures)
    print(f"Calculated mean temperature: {mean_temperature:.2f} K")
    time_steps = np.arange(len(temperatures))

    fig = plt.figure(figsize=(10, 5))
    plt.plot(time_steps, temperatures, color="crimson", linewidth=1, label="Instantaneous T")
    plt.axhline(y=mean_temperature, color="black", linestyle="--", linewidth=1.5, label=f"Mean T ({mean_temperature:.2f} K)")

    plt.xlabel("Frame Index")
    plt.ylabel("Temperature (K)")
    plt.title("Temperature Profile Along the MD Trajectory (at equilibrium)")
    plt.legend()
    plt.grid(True, linestyle=":", alpha=0.6)

    plt.tight_layout()

    # Save the figure instead of plotting it
    filename = "temperature_profile_equilibrium.png"
    save_path = os.path.join(output_dir, filename)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig) # Close the figure to free up memory